# Reproducible Processing and Validation of Wearable Eye-Tracking Data

**USBEREIT 2026 — companion notebook**

This notebook reproduces every figure and table reported in the paper. It
walks through four stages:

1. **Pipeline execution.** Convert raw Pupil Labs Neon recordings into a
   single trial-level table.
2. **Reliability analysis.** Split-half + bootstrap confidence intervals
   on the extracted metrics.
3. **Task-type classification.** Logistic regression and random forest with
   GroupKFold cross-validation by recording.
4. **Configuration sensitivity.** Re-run the pipeline under perturbed
   parameter sets and measure the impact on classifier accuracy and RT
   detection rate.

All processing logic lives in the `src/` package. This notebook is purely an
orchestrator and figure renderer.

**Expected input.** A directory of ZIP archives, one per recording, each
containing a Pupil Labs Neon export. Healthy controls and patients are
distinguished by filename tokens (`HC_*.zip`, `PAT_*.zip`, etc.); manual
overrides are supported through `MANUAL_LABELS`.

**Expected output.** A `figures/` directory with the four figures used in
the paper and a `tables/` directory with the supporting CSVs.


## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the `src` package importable when the notebook is run from `notebooks/`
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import analyses, plotting
from src.config import (
    CFG_DEFAULT,
    PALETTE,
    SENSITIVITY_CONFIGS,
    apply_matplotlib_style,
)
from src.io_utils import build_dataset_index, build_zip_index
from src.pipeline import build_trial_table

apply_matplotlib_style()


### Paths

Set the dataset directory to the folder containing the recording ZIPs. Two
working directories are created automatically: `neon_workdir/` caches the
extracted recordings, and `neon_outputs/` stores intermediate tables.


In [ ]:
DATASET_DIR = Path("neon_zips")          # input ZIPs live here
WORK_DIR    = Path("neon_workdir")       # cached extracted recordings
OUTPUT_DIR  = Path("neon_outputs")       # intermediate CSVs
FIGURES_DIR = ROOT / "figures"           # paper-quality PNG figures
TABLES_DIR  = ROOT / "tables"            # supporting CSV tables

for d in (WORK_DIR, OUTPUT_DIR, FIGURES_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)


## 2. Build the trial table

`build_zip_index` enumerates the ZIPs and assigns a healthy / patient label
from the filename. Pass an explicit override through `MANUAL_LABELS` when a
filename does not match the recognised tokens.


In [ ]:
MANUAL_LABELS: dict[str, int] = {
    # "raw_recording_004.zip": 1,
    # "raw_recording_005.zip": 0,
}

zip_index = build_zip_index(DATASET_DIR, manual_labels=MANUAL_LABELS)
zip_index


In [ ]:
dataset_index = build_dataset_index(zip_index, work_dir=WORK_DIR)
dataset_index


Run the full pipeline with the default configuration. This processes every
recording, extracts per-trial metrics for the five paradigms (Prediction,
Gap, Overlap, Decision, Antisaccade) and concatenates them into a single
table.


In [ ]:
trial_metrics = build_trial_table(
    dataset_index,
    cfg=CFG_DEFAULT,
    output_path=OUTPUT_DIR / "trial_metrics.csv",
)
print(f"trial_metrics shape: {trial_metrics.shape}")
trial_metrics.head()


In [ ]:
# Quick sanity check: trial counts per recording and block.
trial_metrics.groupby(["recording_id", "block"]).size().unstack(fill_value=0)


## 3. Reliability: split-half + bootstrap

For each `(recording_id, block)` pair, trials are split by odd / even
`trial_id`. For every numeric metric we compute the relative difference of
the means and then summarise across pairs with a bootstrap median and 95%
CI (B = 1000 resamples).


In [ ]:
pair_results = analyses.split_half_relative_differences(trial_metrics)
reliability  = analyses.reliability_summary(pair_results, n_iter=1000, seed=42)
reliability.to_csv(TABLES_DIR / "reliability_summary.csv", index=False)
reliability


In [ ]:
fig = plotting.plot_reliability(reliability, output_path=FIGURES_DIR / "fig_reliability.png")
plt.show()


Most metrics show a small median relative difference and narrow CI,
confirming their stability as biosignals across odd / even trial halves.
The two metrics with elevated variability — `express_like` and
`post_response_dx_px` — are interpreted in the paper: the first reflects
genuine inter-individual variation in short-latency saccade rates, the
second is computed from only ~5 gaze samples and is therefore noise-sensitive.


## 4. Task-type classification

GroupKFold cross-validation with `recording_id_num` as the group label
ensures that all trials from one participant stay in the same fold. Two
classifiers are evaluated: a scaled logistic regression and a random forest.

Categorical columns are label-encoded and missing numeric values are imputed
with the column mean (see `analyses.preprocess_fillna`).


In [ ]:
trial_metrics_prep = analyses.preprocess_fillna(trial_metrics)
summary, details = analyses.cross_validate_classifiers(trial_metrics_prep, n_splits=4)
summary.to_csv(TABLES_DIR / "task_classification_summary.csv", index=False)
summary


Confusion matrices for both models, normalised by true class.

In [ ]:
for model_name in ("LogisticRegression", "RandomForest"):
    info = details[model_name]
    fig = plotting.plot_confusion_matrix(
        info["confusion"],
        labels=info["labels"],
        title=f"{model_name}: normalised confusion matrix",
        output_path=FIGURES_DIR / f"fig_confusion_{model_name}.png",
        normalize=True,
    )
    plt.show()


The top-10 informative features for each model. Note that
`trial_duration_ms` and `onset_s` are determined by the stimulus program
rather than oculomotor behaviour; their high importance means the reported
accuracies partially reflect task-structure differences. A purely biological
re-run is left as future work.


In [ ]:
logreg_top, rf_top = analyses.top_features(trial_metrics_prep, k=10)
logreg_top.to_csv(TABLES_DIR / "top_features_logreg.csv", index=False)
rf_top.to_csv(TABLES_DIR / "top_features_rf.csv", index=False)

fig = plotting.plot_feature_importances(
    rf_top, value_col="feature_importance",
    title="Random forest: top-10 feature importances",
    output_path=FIGURES_DIR / "fig_feature_importance_rf.png",
)
plt.show()

fig = plotting.plot_feature_importances(
    logreg_top, value_col="abs_coef",
    title="Logistic regression: top-10 |coefficients|",
    output_path=FIGURES_DIR / "fig_feature_importance_logreg.png",
)
plt.show()


## 5. Configuration sensitivity (ablation)

The pipeline is re-run under eight configurations that scale the RT
detection threshold (`thr_px_ratio`) and the RT hold time
(`hold_ms_reaction`) by different factors. For each configuration we record
the balanced accuracy of a logistic regression and the median per-recording
RT detection rate.

> This is the slowest cell in the notebook: it executes the full pipeline
> eight times. Expect roughly 8× the time of section 2.


In [ ]:
sensitivity = analyses.configuration_sensitivity(
    dataset_index, configs=SENSITIVITY_CONFIGS, n_splits=4,
)
sensitivity.to_csv(TABLES_DIR / "cfg_sensitivity.csv", index=False)
sensitivity


In [ ]:
fig = plotting.plot_cfg_sensitivity(
    sensitivity, output_path=FIGURES_DIR / "fig_cfg_sensitivity.png"
)
plt.show()


## 6. Summary

The notebook has produced:

| Artefact | Location |
|---|---|
| Trial-level metrics | `neon_outputs/trial_metrics.csv` |
| Reliability summary | `tables/reliability_summary.csv` |
| Classification summary | `tables/task_classification_summary.csv` |
| Top features (LR / RF) | `tables/top_features_logreg.csv`, `tables/top_features_rf.csv` |
| Configuration sensitivity | `tables/cfg_sensitivity.csv` |
| Figures (paper-quality PNG) | `figures/*.png` |

These artefacts are the only inputs needed to reproduce the figures and
tables in the paper.
